<h1><center>Laboratorio 9: predecir biomasa en base a imágenes 🌾</center></h1>

<center><strong>IA7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>


---

### Cuerpo Docente

- Profesores: Pablo Badilla e Ignacio Núñez
- Auxiliar: Sofía Chávez
- Ayudantes: Javiera Arévalo, Tamara Cornejo, Ignacio Reyes


### Equipo

- Adolfo Rojas
- Sebastián Carrasco


### Reglas

- Trabajar en equipos de hasta dos personas.
- El extractor de imágenes se usa **solo para inferencia**: nunca llamen `.backward()`, nunca construyan un `optimizer` sobre sus parámetros y nunca muevan un tensor a `cuda`. Este laboratorio corre de principio a fin en un laptop sin GPU, dentro de los dos bloques de la sesión.
- Mantener las transformaciones de tablas en Polars y los pipelines, el escalamiento y la validación cruzada en scikit-learn. La única tabla de pandas es la que devuelve `tabla_runs()`, porque así la entrega MLflow.
- Usar `for` para repetir un experimento (un modelo, un valor de `alpha`, una partición de la validación cruzada) o para recorrer imágenes, lotes y tiles en la Etapa 2. No usarlo para recorrer filas de una tabla: eso se resuelve con expresiones de Polars.
- No modificar los archivos de `data/raw/`.
- No evaluar ningún modelo con `KFold` corriente salvo que la actividad lo pida explícitamente para compararlo con la alternativa agrupada: fuera de esa comparación, toda validación con que se mide un modelo es `GroupKFold`. La validación interna de `RidgeCV` es la excepción, y se discute en la 3.4.


### Flujo de trabajo

El notebook sirve para explorar y decidir. Lo que la evidencia justifica se traslada a `src/pasturecam/`, y las pruebas entregadas indican cuándo una etapa está cerrada:

```bash
uv run pytest -m etapa1
uv run pytest -m etapa2
uv run pytest -m etapa3
uv run pytest -m etapa4
```


<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📌 Instalar antes de la sesión</strong>
Abran una terminal en la raíz de este laboratorio y ejecuten el comando de abajo. Instala <code>torch</code> y <code>torchvision</code> desde el índice CPU de PyTorch (sin ruedas CUDA), además de <code>timm</code>, que trae los modelos preentrenados. El entorno completo pesa cerca de 1,4 GB.

  <table>
    <thead>
      <tr><th>Paquete</th><th>Para qué lo usamos</th><th>Etapa</th></tr>
    </thead>
    <tbody>
      <tr><td><code>polars</code>, <code>pyarrow</code></td><td>Lectura y transformación de la tabla de mediciones</td><td>1</td></tr>
      <tr><td><code>torch</code>, <code>torchvision</code>, <code>timm</code>, <code>pillow</code></td><td>Extractor de imágenes preentrenado, en modo inferencia</td><td>2</td></tr>
      <tr><td><code>scikit-learn</code></td><td>Pipelines, validación cruzada y regresores</td><td>1–4</td></tr>
      <tr><td><code>mlflow</code></td><td>Registro local de experimentos</td><td>1–5</td></tr>
      <tr><td><code>plotly</code></td><td>Gráficos interactivos</td><td>Todas</td></tr>
      <tr><td><code>jupyterlab</code></td><td>Entorno de notebooks</td><td>Todas</td></tr>
    </tbody>
  </table>

La primera vez que corran la Etapa 2, `timm` descarga los pesos de los dos extractores (unos cientos de MB) desde Hugging Face.

</div>

```bash
uv sync --locked --all-groups
```


<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📌 Descargar los datos</strong>
Los datos no vienen en el repositorio. Descarguen <code>mediciones.csv</code> y la carpeta <code>imagenes/</code> desde el enlace que comparte el equipo docente y péguenlos en <code>data/raw/</code>, de modo que queden <code>data/raw/mediciones.csv</code> y <code>data/raw/imagenes/</code>. Sin ellos, las Etapas 1 y 2 no pueden leer nada.

</div>


## Contexto

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📜 El plato medidor de Remigio</strong>

**Remigio Trébol** es agrónomo de la **Cooperativa Lechera del Rahue**, en Osorno. Todas las semanas recorre los potreros de la cooperativa con un <strong>plato medidor</strong>: un disco que se deja caer sobre el pasto y cuya altura de reposo se usa para estimar cuánto forraje hay disponible, y con eso decidir cuántos animales puede soportar cada potrero sin sobrepastorearlo.

  <div style="text-align: center; margin: 16px 0;">
    <img src="static/plato_medidor.jpg" alt="Un hombre de chaleco y lentes está de pie sobre un potrero y sostiene un plato medidor: un disco metálico en la base de una varilla vertical con un contador, apoyado sobre el pasto. Al fondo, una vaca blanca y negra pasta entre árboles." style="width: 450px; max-width: 100%; height: auto;">
  </div>

El método funciona, pero es lento: un potrero grande significa docenas de mediciones, una planilla y bastante caminata. Remigio quiere saber si alcanza con sacarle una foto cenital al pasto desde el celular, sin el plato, para llegar a la misma decisión más rápido.

Antes de comprarle a alguien un sistema hecho a medida —o de financiar una campaña propia de fotos y mediciones en sus potreros—, quiere probar la idea con datos que ya existen. Consiguió un dataset público de CSIRO (la agencia científica australiana): 357 fotos cenitales de cuadrantes de pradera de 70 × 30 cm, cada una con la biomasa seca medida en laboratorio después de cortar y pesar el pasto. El pasto australiano no es idéntico al de Osorno, pero comparte las mismas especies base —ballica y trébol— que los potreros de la cooperativa, así que sirve para la primera prueba.

El encargo es concreto: usando esas fotos y esas mediciones, ¿una imagen puede reemplazar al plato?

</div>


## Objetivos

- Transformar una tabla de mediciones de formato largo a ancho, verificando sus dimensiones antes de modelar.
- Extraer embeddings de imágenes con un extractor preentrenado, en modo inferencia y sin GPU, y entender qué descarta su preprocesamiento.
- Construir un `Pipeline` de escalamiento y regresión regularizada, y detectar cuándo ese pipeline se evalúa de forma optimista.
- Elegir una métrica de regresión en función de la decisión que debe tomar el cliente, no por costumbre.
- Mover un hiperparámetro y observar cómo cambia el desempeño del modelo, como adelanto de la optimización de hiperparámetros.
- Registrar experimentos en un sistema de tracking local y usarlos para fundamentar una recomendación.


### Temas a tratar

Transfer learning con extractores congelados, `timm`, `Pipeline`, Ridge, `GroupKFold` y leakage, MAE/RMSE/R², el hiperparámetro `alpha` de Ridge, y tracking de experimentos con MLflow local.


### Evaluación

<!-- EVALUACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8b5cf6; background:rgba(139,92,246,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📊 Distribución del puntaje</strong>

| Etapa     | Contenido                          | Puntaje |
| --------- | ---------------------------------- | ------: |
| 1         | Datos y baseline del plato medidor |     1,0 |
| 2         | Embeddings preentrenados           |     1,0 |
| 3         | Pipelines y leakage                |     2,0 |
| 4         | Métricas                           |     1,0 |
| 5         | Consultar y decidir                |     1,0 |
| **Total** |                                    | **6,0** |

La nota se calcula como 1,0 + puntaje obtenido.

</div>


## Introducción conceptual

Antes de pasar a los datos, repasemos las ideas que usaremos durante todo el laboratorio. Es solo el concepto: cada una se retoma, con código, donde se necesita.


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Aprendizaje supervisado</strong>

En el Lab8 construimos representaciones y agrupamos personajes sin que ninguna columna dijera "esta es la respuesta correcta". El <strong>aprendizaje supervisado</strong> es distinto: cada ejemplo de entrenamiento trae su <strong>etiqueta</strong>, un valor real que el modelo debe aprender a predecir, y el modelo ajusta sus parámetros para acercarse a ella. Cuando la etiqueta es un número continuo —un precio, una temperatura, un peso— se llama <strong>regresión</strong>; cuando es una categoría, <strong>clasificación</strong>. La diferencia importa porque en una regresión los errores tienen tamaño: predecir 90 cuando el valor real es 100 es un error menor que predecir 10, mientras que en una clasificación una predicción solo acierta o falla.

Si el modelo predice <strong>varios números a la vez</strong> para cada ejemplo, la regresión es <strong>multitarget</strong> (o multisalida): la etiqueta deja de ser un valor y pasa a ser un vector, como cuando se predicen juntas la temperatura y la humedad de mañana.

El diagrama ordena las variedades del aprendizaje supervisado según la etiqueta que se quiere predecir.

```mermaid
flowchart TD
    S["Aprendizaje supervisado<br/>datos con etiqueta → predecir la etiqueta"] --> R["Regresión<br/>la etiqueta es un número continuo"]
    S --> C["Clasificación<br/>la etiqueta es una categoría"]
    R --> R1["Un objetivo<br/>ej.: el precio de una vivienda"]
    R --> R2["Varios objetivos (multitarget)<br/>ej.: temperatura y humedad de mañana"]
    C --> C1["Binaria<br/>ej.: un correo es spam o no"]
    C --> C2["Multiclase<br/>ej.: la especie de una planta"]
```

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Representación de imágenes y embeddings</strong>

Para un computador, una foto es una <strong>matriz de píxeles</strong>: un arreglo de alto × ancho × 3 canales (rojo, verde y azul), con valores entre 0 y 255. Una foto de 1000 × 500 px son 1,5 millones de números, y ninguno de ellos dice algo por sí solo: un píxel aislado no "significa" pasto. Un modelo necesita una <strong>representación</strong>, una forma de describir cada imagen con pocos números que conserven lo importante para la tarea.

Un <strong>embedding</strong> es una representación aprendida: un vector de unos cientos de números que una red neuronal ya entrenada calcula a partir de la imagen, de modo que imágenes parecidas quedan con vectores cercanos e imágenes distintas, con vectores lejanos. En este laboratorio reemplazamos los píxeles por embeddings, y sobre ellos entrenamos un modelo simple.

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Entrenamiento, prueba y métricas</strong>

Un modelo supervisado se ajusta con ejemplos y después se usa con datos que no ha visto. Por eso los datos se separan antes de ajustar: <strong>entrenamiento</strong> (<em>train</em>), con el que el modelo ajusta sus parámetros, y <strong>prueba</strong> (<em>test</em>) o <strong>validación</strong>, que el modelo nunca ve durante el ajuste. El número que importa se calcula sobre los datos de prueba, nunca sobre los de entrenamiento: un modelo puede memorizar sus datos de entrenamiento y fallar por completo con datos nuevos.

Para comparar lo que el modelo predice con el valor real se usa una <strong>métrica de evaluación</strong>, un número que resume qué tan bien lo hace. En regresión miden cuánto se equivoca el modelo (por ejemplo, el error promedio en las unidades del objetivo, o el R²); en clasificación, qué fracción de las veces acierta (la <em>accuracy</em>, o exactitud). Cuál conviene depende de la decisión que se tomará con el resultado.

```mermaid
flowchart LR
    D["Datos con etiqueta"] --> SP{"Separar"}
    SP --> TR["Entrenamiento"]
    SP --> TE["Prueba o validación"]
    TR --> M["Ajustar el modelo"]
    M --> P["Predecir sobre los datos de prueba"]
    TE --> P
    P --> E["Comparar con la etiqueta real<br/>→ métrica de evaluación"]
```

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Tracking de experimentos</strong>

Un cuaderno con decenas de celdas ejecutadas y reescritas es un mal lugar para guardar qué modelo dio qué puntaje: basta reordenar las celdas o correr una de nuevo con otro parámetro para perder la trazabilidad. Un sistema de <strong>tracking</strong> registra cada <strong>run</strong> —una combinación de datos, modelo, hiperparámetros y métricas— de forma permanente y consultable, sin depender de qué celda se ejecutó en qué orden.

<strong>MLflow</strong> es uno de esos sistemas. En su forma local, sin ningún servidor, guarda cada run en una base de datos (acá, SQLite) y sus artefactos —como el modelo ajustado— en el disco. Alcanza con <code>mlflow.set_tracking_uri(...)</code> para decirle dónde. En este laboratorio registraremos cada evaluación desde el primer baseline, y al final usaremos esos registros para decidir.

</div>


## El dataset

Trabajaremos con **CSIRO Image2Biomass**, un dataset público de **357 fotos cenitales de cuadrantes de pradera** de 70 × 30 cm. Cada foto trae la biomasa del pasto de ese cuadrante, medida en laboratorio: se corta el pasto, se seca en horno y se pesa, por eso todas las biomasas son **peso seco, en gramos por cuadrante**. También trae datos del muestreo, como la fecha, el estado y la altura que midió el plato.

| Variable            | Qué es                                                    | Rol                                |
| ------------------- | --------------------------------------------------------- | ---------------------------------- |
| Foto (`image_path`) | Foto cenital del cuadrante, de 1000 × 500 px              | Entrada                            |
| `Sampling_Date`     | Fecha del muestreo (28 fechas distintas)                  | Dato descriptivo                   |
| `State`             | Estado australiano donde se tomó la muestra (4)           | Dato descriptivo                   |
| `Species`           | Especie o mezcla de especies del cuadrante                | Dato descriptivo                   |
| `Pre_GSHH_NDVI`     | Índice de vegetación (NDVI) satelital previo al muestreo  | Dato descriptivo                   |
| `Height_Ave_cm`     | Altura promedio del pasto medida con plato medidor, en cm | Referencia: lo que hoy usa Remigio |
| `Dry_Green_g`       | Biomasa seca de pasto verde, sin trébol                   | **Etiqueta**                       |
| `Dry_Dead_g`        | Biomasa seca de material muerto                           | **Etiqueta**                       |
| `Dry_Clover_g`      | Biomasa seca de trébol                                    | **Etiqueta**                       |
| `GDM_g`             | Materia seca verde (_green dry matter_): verde más trébol | **Etiqueta**                       |
| `Dry_Total_g`       | Biomasa seca total: verde más muerto más trébol           | **Etiqueta** (la principal)        |

Hay cinco etiquetas, y las dos últimas son sumas de las otras. Durante casi todo el laboratorio trabajaremos con **una sola**, `Dry_Total_g`, la biomasa total del cuadrante; en la 4.2 usaremos los cinco objetivos a la vez.


Así se ven los cuadrantes con los que vamos a trabajar. Estas cuatro fotos del dataset van de menos a más biomasa seca total (`Dry_Total_g`), y cada título indica el estado, la especie y las cinco mediciones en gramos. Fíjense en cuánto cambian el color, la densidad y la cantidad de pasto seco de un cuadrante a otro.

<div style="text-align: center; margin: 16px 0;">
  <img src="static/cuadrantes_muestra.jpg" alt="Cuatro fotos cenitales de cuadrantes de pradera, de menos a más biomasa seca total: suelo casi desnudo con marcas rojas (1,0 g), pasto ralo con mucho material seco (40,6 g), pasto mixto con trébol (67,6 g) y pasto verde alto y denso (185,7 g). Cada foto indica estado, especie y las cinco mediciones en gramos." style="width: 800px; max-width: 100%; height: auto;">
</div>


### Rutas del laboratorio y MLflow

Esta celda también deja listo MLflow: la base `mlflow.db` y los artefactos (`mlruns/`) quedan en la carpeta del laboratorio, y todos los runs van al experimento `biomasa-rahue`.


In [ ]:
from pathlib import Path
import sys

RUTA_LAB = Path.cwd().parent
if str(RUTA_LAB) not in sys.path:
    sys.path.insert(0, str(RUTA_LAB))

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

import logging
import os

import mlflow

from src.pasturecam.contratos import NOMBRE_EXPERIMENTO

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
logging.getLogger("mlflow").setLevel(logging.ERROR)
mlflow.set_tracking_uri(f"sqlite:///{RUTA_LAB}/mlflow.db")
if mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO) is None:
    mlflow.create_experiment(
        NOMBRE_EXPERIMENTO, artifact_location=(RUTA_LAB / "mlruns").as_uri()
    )
mlflow.set_experiment(NOMBRE_EXPERIMENTO)

---

## Etapa 1 — Datos y baseline del plato medidor (1,0 puntos)


<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📜 "¿Cuánto se equivoca hoy mi plato?"</strong>

Remigio entrega la planilla y las fotos sin más instrucciones que una pregunta: antes de hablar de inteligencia artificial, quiere saber qué tan bien funciona lo que ya usa. Si una foto va a reemplazar al plato medidor, primero hay que saber contra qué compite.

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Unidad de observación y formato largo</strong>

La <strong>unidad de observación</strong> es lo que representa cada fila de una tabla. Una tabla de ventas puede traer una fila por transacción, por producto o por día, según cómo se construyó. Antes de transformar cualquier tabla conviene preguntarse qué identifica a una fila y cuántas filas distintas se esperan.

Una tabla en <strong>formato largo</strong> repite esa identidad en varias filas, una por cada variable medida. Por ejemplo, el historial clínico de un paciente puede traer una fila por paciente y control (presión, temperatura, peso), en vez de una columna por variable. Pasarla a <strong>formato ancho</strong> —una fila por paciente, una columna por variable— es necesario para muchos modelos, pero exige cuidado: si dos filas "largas" distintas deberían fusionarse en la misma fila "ancha" y no lo hacen, se duplica la unidad de observación.

</div>


`data/raw/mediciones.csv` trae la tabla tal como se publicó, en formato largo: cada fotografía aparece en **cinco filas**, una por cada objetivo que se midió en laboratorio. Carguémosla y miremos su forma antes de tocar nada.


In [ ]:
import polars as pl

mediciones_largo = pl.read_csv(RAW_DIR / "mediciones.csv")
mediciones_largo

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Antes de pivotear</strong>

1. ¿Qué identifica a una fila de `mediciones_largo`: la imagen, el objetivo, o la combinación de ambos?
2. Si la tabla tiene 357 imágenes y 5 objetivos por imagen, ¿cuántas filas debería tener `mediciones_largo`, y cuántas debería tener la versión ancha (una fila por imagen)?
 </div>


1. El sample*id que está compuesto por ambos, el filename de la imagen + "*" + objetivo puesto que la imagen se repite en los 5 registros y el diferenciador es el objetivo/etiqueta
2. El formato largo tendrá 357\*5=1785 registros mientras que la versión densa/ancha 357


`target_name` nombra el objetivo de cada fila (`Dry_Green_g`, `Dry_Dead_g`, `Dry_Clover_g`, `GDM_g` o `Dry_Total_g`, todos en gramos) y `target` trae su valor. El resto de las columnas —`image_path`, `Sampling_Date`, `State`, `Species`, `Pre_GSHH_NDVI` y `Height_Ave_cm`— se repite igual en las cinco filas de una misma imagen.


### 1.1 — De formato largo a ancho (0,3 puntos)

Pasemos `mediciones_largo` a formato ancho: una fila por imagen y una columna por cada objetivo. Háganlo primero en el notebook, probando de forma interactiva, y guárdenlo en `mediciones_ancho`. Usen como índice todas las columnas salvo `sample_id`, `target_name` y `target`: `sample_id` es distinto en cada una de las cinco filas de una imagen y, si entrara al índice, esas filas no se juntarían. El resultado debe tener 357 filas y 11 columnas: seis de datos de la imagen y cinco objetivos.

(Hint: `pivot`)


In [ ]:
columnas_indice = [
    c for c in mediciones_largo.columns if c not in ("sample_id", "target_name", "target")
]

display(mediciones_largo.head(5))

# pivot de largo a ancho
mediciones_ancho = mediciones_largo.pivot(
    index=columnas_indice, # row identifiers
    on="target_name", # new columns
    values="target" # the values to fill the new columns with
)
display(mediciones_ancho)

Ya que la exploración funciona, vale la pena llevarla a una función que se pueda reutilizar y probar.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `data.py`</strong>

Implementen en <code>src/pasturecam/data.py</code>:

  <ul>
    <li><code>a_formato_ancho(df_largo)</code>: la misma operación que acaban de explorar. Pivotea sobre <code>target_name</code>/<code>target</code>, usando el resto de las columnas (menos <code>sample_id</code>) como índice, y devuelve una fila por imagen con una columna por cada nombre en <code>OBJETIVOS</code>.</li>
  </ul>
</div>


In [ ]:
from src.pasturecam.data import a_formato_ancho

mediciones = a_formato_ancho(mediciones_largo)
print(f"Exploración en el notebook: {mediciones_ancho.shape}")
print(f"Función del módulo:        {mediciones.shape}")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
Ejecutar <code>uv run pytest -m etapa1</code>. Las dos formas impresas arriba deben ser <code>(357, 11)</code>: la función del módulo reproduce lo que explorarían en el notebook.

</div>


### 1.2 — Distribución y transformación logarítmica (0,2 puntos)

Antes de modelar conviene mirar cómo se distribuye lo que queremos predecir, porque de eso depende cómo se comporta cualquier error que midamos sobre ello. Si casi todos los cuadrantes pesan parecido y unos pocos pesan muchísimo, esos pocos terminan dominando los promedios y las métricas. Observen la distribución de `Dry_Total_g` y fíjense en dos cosas: dónde se concentra la mayoría de los cuadrantes y hasta dónde llegan los más cargados.


In [ ]:
import plotly.express as px

px.histogram(mediciones, x="Dry_Total_g", nbins=40, title="Dry_Total_g (g)")

La mitad central de los cuadrantes pesa entre 25 y 58 g, pero la cola llega hasta casi 190 g: muchos cuadrantes livianos y pocos muy cargados. A eso se le llama asimetría a la derecha, y es lo que la siguiente transformación ayuda a manejar.


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Asimetría y `log1p`</strong>

Una variable con <strong>asimetría a la derecha</strong> tiene una cola larga de valores altos y poco frecuentes: el precio de las viviendas de una ciudad, los ingresos de un hogar, el número de visitas a una página web. Una métrica calculada directamente sobre esa escala le da demasiado peso a los pocos casos extremos.

La transformación <code>log1p(x) = log(1 + x)</code> comprime esa cola sin romperse en <code>x = 0</code> (a diferencia de <code>log(x)</code>, que no está definido ahí) y es invertible con <code>expm1</code>. Es razonable cuando los valores son no negativos y la variable es asimétrica — exactamente la forma de una distribución de biomasa, que nunca es negativa y tiene algunos cuadrantes mucho más cargados que el resto.

En este laboratorio usaremos <code>log1p</code> para <strong>evaluar</strong>: la métrica <code>r2_log</code> compara lo real y lo predicho después de transformar ambos con <code>log1p</code>, de modo que mide el error en términos relativos. Los modelos, en cambio, se ajustan en gramos.

</div>


La tabla cuenta, para cada objetivo, cuántos cuadrantes valen exactamente 0 g.


In [ ]:
import numpy as np

mediciones.select(
    pl.col("Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g")
    .eq(0)
    .sum()
)

print(f"La diferencia en log: {np.log1p(30) - np.log1p(20):.3f} y {np.log1p(210) - np.log1p(200):.3f}")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Por qué `log1p` y no `log`</strong>

1. Si usáramos `log(x)` en vez de `log1p(x)`, ¿qué pasaría con los cuadrantes que valen 0 g? Según la tabla de arriba, ¿cuál de los cinco objetivos se vería más afectado?
2. Un modelo predice 30 g donde lo real es 20 g, y otro predice 210 g donde lo real es 200 g: en los dos casos el error es de 10 g. Calculen la diferencia de `log1p` en cada caso (`np.log1p(30) - np.log1p(20)` y `np.log1p(210) - np.log1p(200)`). ¿Cuál es mayor, y qué dice eso de cómo `log1p` mide el error?
 </div>


1. Porque si bien algunos objetivos como GDM_g o Dry_Total_g no se vean afectados, el resto (y en especial los cuadrantes sin treboles Dry_Clover_g=0 que representan más de 1/3) al aplicarle la transformación log() por convención de numpy arrojaría -inf, arruinando los datos y las métricas que calculemos
2. La diferencia para el primer caso (~0.39) es mucho mayor que el del segundo (~0.05) a pesar de ser la misma diferencia en gramos, con esto podemos decir que el error se mide de forma relativa (10 de 20 es harto pero 10 de 200 casi despreciable)


### 1.3 — Grupos de muestreo (0,1 puntos)

Las fotos tomadas en una misma fecha o en un mismo estado comparten condiciones de muestreo, y más adelante vamos a necesitar tratarlas como grupos. Cuenten cuántas imágenes hay por cada valor de `Sampling_Date` y por cada valor de `State`, y guarden los conteos en `por_fecha` y `por_estado`: deben quedar 28 y 4 filas.


In [ ]:
por_fecha = (
    mediciones.group_by("Sampling_Date")
    .agg(pl.len().alias("n_imagenes"))
    .sort("n_imagenes", descending=True)
)
por_estado = (
    mediciones.group_by("State")
    .agg(pl.len().alias("n_imagenes"))
    .sort("n_imagenes", descending=True)
)

display(por_fecha)
display(por_estado)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Elegir el grupo</strong>

1. En la Etapa 3 vamos a repartir las fotos en 5 particiones, con todas las fotos de un mismo grupo en la misma partición. ¿Cuál de las dos variables permite formar 5 particiones de tamaño parecido, y por qué?
 </div>


`Escriban su respuesta aquí:`


### 1.4 — Regresión lineal, ajuste y R² (0,2 puntos)

En la 1.5 vamos a entrenar los primeros modelos del laboratorio. Antes, veamos qué hace una regresión lineal, qué significa ajustarla y cómo se mide qué tan bien lo hace.


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Regresión lineal, ajuste y R²</strong>

Una <strong>regresión lineal</strong> predice un número como una suma ponderada de las variables de entrada: $\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_d x_d + b$. Cada $x_j$ es una variable, $w_j$ es el peso que el modelo le da, $b$ es el intercepto (lo que predice cuando todas las variables valen 0) y $\hat{y}$ es la predicción. Con una sola variable es una recta; con $d$ variables, un hiperplano.

<strong>Ajustar</strong> el modelo es elegir los pesos y el intercepto para que las predicciones queden lo más cerca posible de las mediciones reales de entrenamiento. La distancia vertical entre un punto y la recta, $y_i - \hat{y}_i$, es el <strong>residuo</strong>. El método de mínimos cuadrados elige la recta que minimiza la suma de los residuos al cuadrado: al elevarlos, los positivos y los negativos no se cancelan y los errores grandes pesan más.

El <strong>R²</strong> compara ese error con el de una referencia simple, predecir siempre el promedio $\bar{y}$ de las mediciones:

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

El numerador es el error del modelo y el denominador, el de la referencia. Vale 1 si el modelo no se equivoca nunca y 0 si no mejora al promedio. Medido sobre datos que el modelo no usó para ajustarse, puede ser negativo: eso significa que predice peor que el promedio.

Para ganar intuición, la figura muestra seis rectas distintas sobre una misma nube de puntos de otro dominio: el precio de viviendas (<code>MEDV</code>) contra su número de habitaciones (<code>RM</code>), ambos estandarizados. Las rectas que siguen la tendencia de los puntos, como las de R² cercano a 0,48, dejan residuos chicos y tienen el R² más alto. Las que se apartan de ella dejan residuos grandes y su R² baja. La última, casi horizontal, predice prácticamente el mismo valor para todas las viviendas, casi el promedio, y su R² es casi 0.

  <div style="text-align: center; margin: 16px 0;">
  <img src="static/r2_ajuste_rectas.jpg" alt="Seis gráficos de dispersión con la misma nube de puntos, precio de viviendas contra número de habitaciones, ambos estandarizados, cada uno con una recta distinta. Las rectas más inclinadas, que siguen la tendencia de los puntos, tienen R² de 0,482 y 0,483; las menos inclinadas tienen R² de 0,297, 0,312 y 0,308, y una recta casi horizontal tiene R² de 0,047." style="width: 700px; max-width: 100%; height: auto;">
  </div>
</div>


Veamos las dos ideas con nuestro dataset: `Height_Ave_cm` contra `Dry_Total_g`, la regresión que Remigio va a evaluar en la 1.5. Cada segmento rojo es un residuo; a la izquierda se predice siempre el promedio y a la derecha, la recta ajustada.

Un aviso sobre el R² de este gráfico: se ajusta y se mide sobre **todos** los datos, y en gramos. En la 1.5 lo vamos a comparar con otra forma de medir.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

altura_demo = mediciones["Height_Ave_cm"].to_numpy()
total_demo = mediciones["Dry_Total_g"].to_numpy()

modelo_demo = LinearRegression().fit(altura_demo.reshape(-1, 1), total_demo)
pred_recta = modelo_demo.predict(altura_demo.reshape(-1, 1))
pred_promedio = np.full_like(total_demo, total_demo.mean())
print(
    f"Pendiente: {modelo_demo.coef_[0]:.2f} g por cm; "
    f"intercepto: {modelo_demo.intercept_:.1f} g"
)

fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=[
        f"Siempre el promedio: R² = {r2_score(total_demo, pred_promedio):.2f}",
        f"Recta ajustada: R² = {r2_score(total_demo, pred_recta):.2f}",
    ],
)
n = len(altura_demo)
orden = np.argsort(altura_demo)
for col, prediccion in enumerate([pred_promedio, pred_recta], start=1):
    x_residuos = np.column_stack([altura_demo, altura_demo, np.full(n, np.nan)]).ravel()
    y_residuos = np.column_stack([total_demo, prediccion, np.full(n, np.nan)]).ravel()
    fig.add_trace(
        go.Scatter(x=x_residuos, y=y_residuos, mode="lines", line=dict(color="#f85149", width=1)),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(x=altura_demo, y=total_demo, mode="markers", marker=dict(color="#58a6ff", opacity=0.6)),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(x=altura_demo[orden], y=prediccion[orden], mode="lines", line=dict(color="#8b949e", width=3)),
        row=1,
        col=col,
    )
fig.update_xaxes(title_text="Height_Ave_cm (cm)")
fig.update_yaxes(title_text="Dry_Total_g (g)", col=1)
fig.update_layout(showlegend=False, height=380, margin=dict(t=60, b=10))
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Leer el R²</strong>

1. En la figura de las seis rectas, la última es casi horizontal y su R² es casi 0. ¿Por qué una recta completamente horizontal, ubicada a la altura del promedio, tendría un R² exactamente igual a 0?
2. Si a la regresión de nuestro gráfico le agregáramos una segunda variable, por ejemplo `Pre_GSHH_NDVI`, ¿podría bajar el R² medido sobre los mismos datos con que se ajusta? ¿Qué dice eso de usar ese R² para decidir si una variable ayuda a predecir?
 </div>


1. Porque en la formula del R^2 al reemplazar $\hat{y}_i = \bar{y}$, el numerador (la suma de cuadrados residual $\sum_i (y_i - \hat{y}_i)^2$) se iguala al denominador (la suma de cuadrados total $\sum_i (y_i - \bar{y})^2$) quedando $R^2 = 1-1$
2. No, pasa algo similar al p-hacking (en este caso overfitting puesto que el modelo se vuelve más complejo y adapta mejor a los datos de entrenamiento), en el peor de los casos el R^2 que se obtenga sea exactamente el mismo al primer modelo si es que el peso de la variable (el coef. del regresor) fuese 0. Por esto no es una métrica útil para determinar el aporte marginal predictivo de agregar una variable o no


### 1.5 — Dos baselines: la media y la altura del plato (0,2 puntos)

Antes de pensar en fotos, Remigio quiere un número: cuánto se equivoca hoy.


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Baseline</strong>

Un <strong>baseline</strong> (o línea base) es un modelo de referencia, simple a propósito, contra el cual se compara cualquier modelo más elaborado. Responde la pregunta «¿cuánto mejor es esto que lo más simple que podríamos hacer?». Sin un baseline, un puntaje aislado no dice mucho: un R² de 0,3 puede ser un gran avance o no mejorar en nada lo que ya se hacía.

Suelen usarse dos tipos. El <strong>baseline trivial</strong> no aprende nada de las variables: para pronosticar la temperatura de mañana, repetir la de hoy; para clasificar correos, decir siempre «no es spam». Marca el piso: un modelo que no lo supera no aprendió nada útil. El <strong>baseline de la práctica actual</strong> es lo que hoy se usa para resolver el problema; superarlo es lo que justifica cambiar de método.

</div>


Usaremos un baseline de cada tipo:

- **La media (trivial).** Para cada imagen de validación se predice el promedio de `Dry_Total_g` calculado solo con las imágenes de entrenamiento (unos 45 g), sin mirar la foto ni ninguna otra columna. Es lo que predeciría alguien que no conoce el potrero.
- **La altura del plato (práctica actual).** Una regresión lineal de `Dry_Total_g` sobre `Height_Ave_cm`, la altura que mide el propio plato. Es el método que Remigio usa hoy.

Ambos se evalúan con el R² sobre `log1p` de la 1.2 y con **validación cruzada** (_cross-validation_): las fotos se reparten en 5 particiones, el modelo se ajusta cinco veces, cada vez dejando afuera una partición distinta para medir, y se promedian los cinco puntajes. Repetir la separación evita depender de una sola partición, que puede salir optimista o pesimista por azar. Usaremos `KFold` con `shuffle=True`, que asigna cada foto a una partición al azar (todavía sin agrupar; eso se revisa en la Etapa 3).

Un R² de 0 significa «igual de bien que predecir siempre el promedio», así que el baseline de la media no puede quedar por encima de 0 salvo por azar. En la práctica queda bajo 0 por dos razones: el promedio de entrenamiento no coincide con el de validación y, como medimos sobre `log1p`, predecir el promedio en gramos (unos 45 g) no equivale a predecir el promedio en escala logarítmica, que es más bajo por la cola de cuadrantes cargados.


In [ ]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

from src.pasturecam.contratos import ResultadoCV

y_total = mediciones["Dry_Total_g"].to_numpy()
altura = mediciones["Height_Ave_cm"].to_numpy().reshape(-1, 1)


def r2_log(y_real, y_pred):
    return r2_score(np.log1p(y_real), np.log1p(np.clip(y_pred, 0, None)))


def evaluar_simple(X, y, modelo_fn, cv):
    puntajes = []
    for train, test in cv.split(X, y):
        modelo = modelo_fn().fit(X[train], y[train])
        puntajes.append(r2_log(y[test], modelo.predict(X[test])))
    return ResultadoCV(
        puntajes=puntajes,
        promedio=float(np.mean(puntajes)),
        desviacion=float(np.std(puntajes)),
        descripcion_validacion=f"KFold({cv.get_n_splits()})",
    )


# cv = cross-validation (validación cruzada): 5 particiones al azar.
cv_simple = KFold(n_splits=5, shuffle=True, random_state=0)
resultado_media = evaluar_simple(
    altura, y_total, lambda: DummyRegressor(strategy="mean"), cv_simple
)
resultado_altura = evaluar_simple(
    altura, y_total, LinearRegression, cv_simple
)
print(f"Media (Dummy):        R2_log = {resultado_media.promedio:.3f} +/- {resultado_media.desviacion:.3f}")
print(f"Altura (plato):       R2_log = {resultado_altura.promedio:.3f} +/- {resultado_altura.desviacion:.3f}")

Estos dos números son los primeros que vamos a querer comparar más adelante, así que los registramos en MLflow desde ya. Registren a mano el baseline de la media como un run llamado `"baseline-media_kfold"`, con:

- los parámetros `modelo="DummyRegressor"` y `entrada="ninguna"`;
- los tags `tipo="baseline"` y `validacion=resultado_media.descripcion_validacion`;
- las métricas `promedio` y `desviacion` de `resultado_media`.

La celda de rutas ya dejó activo el experimento `biomasa-rahue`, así que el run queda ahí.

(Hint: `mlflow.start_run(run_name=...)`, `mlflow.log_params`, `mlflow.set_tags`, `mlflow.log_metrics`)


In [ ]:
with mlflow.start_run(run_name="baseline-media_kfold"):
    mlflow.log_params({"modelo": "DummyRegressor", "entrada": "ninguna"})
    mlflow.set_tags(
        {
            "tipo": "baseline",
            "validacion": resultado_media.descripcion_validacion,
        }
    )
    mlflow.log_metrics(
        {
            "promedio": resultado_media.promedio,
            "desviacion": resultado_media.desviacion,
        }
    )

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📌 `registrar_run` y `tabla_runs` vienen implementadas</strong>
<code>src/pasturecam/tracking.py</code> es material provisto: no hay que completarlo. <code>registrar_run(nombre, resultado, params, modelo=None, etiquetas=None)</code> hace lo mismo que acaban de hacer a mano, para cualquier <code>ResultadoCV</code>: registra los parámetros, la validación y las <code>etiquetas</code> como tags, el puntaje de cada fold, el promedio y la desviación, y, si se le entrega, el modelo ajustado como artefacto. <code>tabla_runs()</code> devuelve los runs del experimento, uno por nombre: si ejecutan el notebook dos veces, se queda con el más reciente. Desde aquí registramos cada evaluación con estas dos funciones.

</div>


In [ ]:
from src.pasturecam.tracking import registrar_run, tabla_runs

registrar_run(
    "baseline-altura_kfold",
    resultado_altura,
    {"modelo": "LinearRegression", "entrada": "Height_Ave_cm"},
    modelo=LinearRegression().fit(altura, y_total),
    etiquetas={"tipo": "baseline"},
)

columnas_runs = [
    "tags.mlflow.runName",
    "tags.tipo",
    "tags.validacion",
    "metrics.promedio",
    "metrics.desviacion",
]
tabla_baselines = tabla_runs()
tabla_baselines[
    (tabla_baselines["tags.tipo"] == "baseline")
    & (tabla_baselines["tags.validacion"] == "KFold(5)")
][columnas_runs]

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
El baseline de altura debe superar al de la media (un R²_log más alto, cercano a 0,10 frente a uno negativo), pero ninguno de los dos es un número para presentarle a un directorio todavía: esta partición no es la que usaremos de aquí en adelante. La tabla de MLflow debe mostrar los dos runs, `baseline-media_kfold` y `baseline-altura_kfold`, con los mismos promedios impresos más arriba.

</div>


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Un número provisional</strong>

1. El R²_log de la altura mejora sobre predecir siempre el promedio, pero sigue siendo bajo en términos absolutos, lejos de 1,0. ¿Esto significa que el plato medidor es un mal instrumento en el potrero, o que predecir la biomasa de un cuadrante solo a partir de una altura promedio es una tarea difícil? ¿Son lo mismo?
2. El R² de la recta del gráfico de la 1.4 (0,25) es más alto que el R²_log de la altura en esta actividad, aunque es la misma regresión. ¿Qué cambió en la forma de medir? ¿Cuál de esos cambios creen que pesa más? Pueden comprobarlo copiando `evaluar_simple` con `r2_score` en vez de `r2_log`.
 </div>


`Escriban sus respuestas aquí:`


---

Con esto Remigio tiene un primer número, aunque todavía provisional: la Etapa 3 lo recalcula con la partición correcta antes de compararlo en serio contra una foto.


---

## Etapa 2 — Embeddings preentrenados (1,0 puntos)


<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📜 "Las redes ya saben mirar fotos"</strong>

Remigio le mostró el número de la Etapa 1 a su sobrino, que estudia ingeniería. La respuesta fue inmediata: "tío, eso ya lo resuelve una red neuronal, ni hay que entrenar nada, bajas un modelo y listo". Remigio no le cree tan fácil, pero tampoco tiene cómo discutirlo. Les pide probarlo con las mismas 357 fotos.

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Transfer learning y extractor congelado</strong>

Entrenar una red de visión desde cero necesita millones de imágenes etiquetadas y horas de GPU. El <strong>transfer learning</strong> evita eso: se toma una red ya entrenada en una tarea masiva —por ejemplo, clasificar un millón de fotos en mil categorías— y se reutiliza su conocimiento para un problema distinto y mucho más chico.

La forma más simple de reutilizarla es como <strong>extractor congelado</strong>: se le pasa una imagen, se recupera la representación numérica que la red calculó justo antes de su capa de clasificación final —el <strong>embedding</strong>— y se descarta el resto. La red nunca se ajusta ("congelada": sin gradientes, sin <code>optimizer</code>, solo inferencia); lo único que se entrena después es un modelo simple sobre esos embeddings.

Cada red espera sus imágenes preprocesadas de una forma específica: un tamaño fijo, una normalización de colores, a veces un recorte. Usar el preprocesamiento equivocado no tira un error — tira un embedding igual de válido en forma, pero que no significa lo que la red aprendió a representar.

</div>


### ¿Por qué un embedding sirve para una regresión lineal?

Una regresión lineal necesita una fila de números por observación. Una foto de 1000 × 500 px con tres canales son 1,5 millones de números, y una suma ponderada de píxeles no aprende casi nada útil: si el pasto se corre unos centímetros, cada píxel cambia de valor y los pesos dejan de calzar. El embedding resuelve eso: resume la foto en 384 o 512 números que la red ya organizó para que fotos visualmente parecidas queden con vectores parecidos, y sobre esos números una suma ponderada sí puede aprender qué componentes se asocian con más pasto.

**Ventajas**

- Cada foto pasa a ser una fila de una tabla, y se puede usar todo lo que ya conocemos: `Pipeline`, `GroupKFold`, regresores de scikit-learn.
- Se pasa de millones de valores a unos cientos.
- El embedding se extrae una sola vez y se guarda en caché; después, entrenar la regresión toma segundos y no necesita GPU.

**Desventajas**

- La red nunca vio biomasa: lo que importa para predecirla (densidad, pasto seco) puede no estar en el embedding, o estar mezclado con otras cosas.
- Cada componente es un número sin significado propio, así que el modelo no se puede explicar en términos de pasto.
- Siguen siendo más componentes (384 o 512) que fotos (357), por lo que el modelo sobreajusta con facilidad; de eso se encarga la regularización de Ridge en la Etapa 3.
- Una regresión lineal solo captura relaciones lineales entre los componentes y la biomasa.


Usaremos dos extractores de `timm`, ambos congelados y en modo inferencia: `resnet18`, que es el punto de comparación, y `vit_small_patch14_dinov2`, el extractor principal del laboratorio. La tabla reúne en un solo lugar cómo se entrenó cada uno y qué se gana y se pierde al usarlo, para volver a ella cuando comparemos resultados.

|                         | `resnet18`                                                                                                                    | `vit_small_patch14_dinov2`                                                                                                                                            |
| ----------------------- | ----------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Arquitectura            | Red convolucional (CNN)                                                                                                       | Vision Transformer pequeño (ViT-S/14)                                                                                                                                 |
| Preentrenamiento        | Supervisado: clasificar las 1000 categorías de ImageNet                                                                       | Autosupervisado (DINOv2): sin etiquetas                                                                                                                               |
| Parámetros              | unos 11 M                                                                                                                     | unos 22 M                                                                                                                                                             |
| Tamaño de entrada       | 224 × 224 px                                                                                                                  | 518 × 518 px                                                                                                                                                          |
| Dimensión del embedding | 512                                                                                                                           | 384                                                                                                                                                                   |
| Ventajas                | Liviano y rápido: procesa imágenes chicas con pocos parámetros. Es una referencia estándar contra la cual comparar.           | Aprendió textura y estructura visual sin que nadie le dijera qué había en cada imagen, lo que lo hace fuerte para fotos de un dominio que nunca vio, como un potrero. |
| Desventajas             | Aprendió a distinguir categorías de objetos, no densidad ni textura fina del pasto, y eso puede quedar fuera de su embedding. | Más lento y más pesado: procesa más de cinco veces más píxeles por imagen con el doble de parámetros.                                                                 |

Ambos preprocesan con un recorte central cuadrado, así que ninguno ve la imagen completa; la 2.3 mide cuánto se pierde. La tercera variante que compararemos en la Etapa 3 no es un modelo distinto: es el mismo DINOv2 con otra forma de presentarle la foto, que construiremos en la 2.3.


### 2.1 — Qué descarta el preprocesamiento (0,2 puntos)

Antes de medir cuánto se pierde en la 2.3, miremos qué le hace el preprocesamiento de un extractor a una foto del dataset: la del pasto alto y denso de la introducción (`ID384648061`).

Antes de ejecutar la celda, investiguen en la documentación de `torchvision` qué hacen las transformaciones `Resize` y `CenterCrop`: las dos aparecen en la transformación que imprime la celda.


In [ ]:
import timm
from PIL import Image

extractor_demo = timm.create_model("resnet18", pretrained=True, num_classes=0)
extractor_demo.eval()
config_demo = timm.data.resolve_data_config({}, model=extractor_demo)
transformacion_demo = timm.data.create_transform(**config_demo)
print(transformacion_demo)

foto_demo = Image.open(RAW_DIR / "imagenes" / "ID384648061.jpg")
print(f"Tamaño original: {foto_demo.size}")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Antes de mirar el cuadrante</strong>

1. Con lo que investigaron, expliquen en una frase qué hace `Resize` y qué hace `CenterCrop`. En particular, ¿`Resize` con un solo número deforma la imagen o conserva su proporción?
2. La transformación de arriba hace un `Resize` y después un `CenterCrop`. Si una imagen es más ancha que alta, ¿qué eje pierde más información con un recorte central cuadrado: el ancho o el alto?
 </div>


`Escriban sus respuestas aquí:`


Comprobémoslo en la foto: a la izquierda, la original; a la derecha, lo que recibe `resnet18` después del `Resize` y el `CenterCrop`.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torchvision.transforms import Compose


def mostrar_imagenes(imagenes, titulos):
    fig = make_subplots(rows=1, cols=len(imagenes), subplot_titles=titulos)
    for i, imagen in enumerate(imagenes, start=1):
        fig.add_trace(go.Image(z=np.asarray(imagen.convert("RGB"))), row=1, col=i)
        fig.update_xaxes(visible=False, constrain="domain", row=1, col=i)
        fig.update_yaxes(
            visible=False,
            scaleanchor=f"x{i}" if i > 1 else "x",
            constrain="domain",
            row=1,
            col=i,
        )
    fig.update_layout(height=320, margin=dict(t=50, b=10, l=10, r=10))
    fig.show()


recorte_demo = Compose(transformacion_demo.transforms[:2])(foto_demo)
mostrar_imagenes(
    [foto_demo, recorte_demo],
    [
        "Foto original",
        f"Lo que ve resnet18 ({recorte_demo.width} × {recorte_demo.height})",
    ],
)

### 2.2 — Extraer embeddings de las 357 fotos (0,4 puntos)

`timm.create_model(nombre, pretrained=True, num_classes=0)` devuelve el extractor sin su capa de clasificación. `resolve_data_config` y `create_transform` reconstruyen el preprocesamiento exacto con el que se entrenó ese modelo — nunca se inventa a mano.

Antes de procesar las 357 fotos, extraigan en el notebook el embedding de las primeras cuatro, con el extractor `resnet18` de la 2.1. Para cada foto de `rutas_imagenes[:4]`: abrirla, convertirla a RGB, aplicarle `transformacion_demo` y juntar las cuatro en un solo lote; después pasar el lote por `extractor_demo` **sin calcular gradientes** y pasar el resultado a NumPy. Guárdenlo en `embeddings_prueba`, que debe tener forma `(4, 512)`: una fila por foto y 512 dimensiones, las del extractor.

(Hint: `torch.stack`, `torch.no_grad`)


In [ ]:
nombres_archivo = mediciones["image_path"].str.replace(
    "imagenes/", "", literal=True
)
rutas_imagenes = [RAW_DIR / "imagenes" / p for p in nombres_archivo.to_list()]

In [ ]:
# Su código aquí
raise NotImplementedError("Construyan embeddings_prueba antes de continuar.")

Para las 357 fotos hace falta repetir esto por lotes, sin quedarse sin memoria, y reutilizarlo con otros extractores. Vale la pena llevarlo a una función.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `embeddings.py`</strong>

Implementen en <code>src/pasturecam/embeddings.py</code> <code>extraer_embeddings(rutas, modelo, transformacion, tamano_lote=16)</code>: la misma operación que acaban de explorar, pero para cualquier lista de rutas. Abre cada imagen de <code>rutas</code>, le aplica <code>transformacion</code>, agrupa en lotes de <code>tamano_lote</code> y pasa cada lote por <code>modelo</code> sin calcular gradientes. Devuelve un arreglo <code>(len(rutas), dimensión)</code>, en el mismo orden que <code>rutas</code>. <code>modelo</code> puede ser cualquier invocable sobre un tensor por lotes — en este laboratorio, un extractor de <code>timm</code> en modo evaluación.

</div>


Extraer embeddings de las 357 fotos toma minutos, no segundos. Los guardamos en `data/processed/` —un output generado, fuera de Git— y, si el notebook se vuelve a correr, los reutilizamos en vez de recalcularlos. Si corrigen `extraer_embeddings` o `dividir_en_tiles` después de una primera ejecución, borren los archivos `data/processed/embeddings_*.parquet` antes de volver a correr estas celdas: si no, el notebook seguirá usando los embeddings anteriores.


In [ ]:
import time

import numpy as np

from src.pasturecam.embeddings import extraer_embeddings

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def guardar_embeddings(embeddings, ruta):
    columnas = [f"d{i}" for i in range(embeddings.shape[1])]
    pl.DataFrame(embeddings, schema=columnas, orient="row").write_parquet(ruta)


def leer_embeddings(ruta):
    return pl.read_parquet(ruta).to_numpy()


ruta_cache_resnet = PROCESSED_DIR / "embeddings_resnet18_centro.parquet"
if ruta_cache_resnet.exists():
    embeddings_resnet = leer_embeddings(ruta_cache_resnet)
else:
    embeddings_resnet = extraer_embeddings(
        rutas_imagenes, extractor_demo, transformacion_demo
    )
    guardar_embeddings(embeddings_resnet, ruta_cache_resnet)
print(f"Forma de los embeddings: {embeddings_resnet.shape}")
print(
    "Las primeras cuatro filas coinciden con la exploración: "
    f"{np.allclose(embeddings_resnet[:4], embeddings_prueba, atol=1e-4)}"
)

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
Ejecutar <code>uv run pytest -m etapa2 -k extraer_embeddings</code>. La celda de arriba debe mostrar <code>(357, 512)</code>: una fila por imagen, en el mismo orden que <code>mediciones</code>, con la dimensión que declara <code>resnet18</code>, y la comparación con la exploración debe dar <code>True</code>.

</div>


Repitamos con el extractor principal del laboratorio: `vit_small_patch14_dinov2`, preentrenado de forma autosupervisada — sin que nadie le dijera qué había en cada imagen, aprendió a representar textura y estructura visual, lo que lo hace fuerte para extraer features congeladas sobre fotos de un dominio que nunca vio en entrenamiento, como un potrero.


In [ ]:
dinov2 = timm.create_model(
    "vit_small_patch14_dinov2", pretrained=True, num_classes=0
)
dinov2.eval()
config_dinov2 = timm.data.resolve_data_config({}, model=dinov2)
transformacion_dinov2 = timm.data.create_transform(**config_dinov2)
print(f"input_size: {config_dinov2['input_size']}")

ruta_cache_dinov2 = PROCESSED_DIR / "embeddings_dinov2_centro.parquet"
if ruta_cache_dinov2.exists():
    embeddings_dinov2 = leer_embeddings(ruta_cache_dinov2)
else:
    t0 = time.time()
    embeddings_dinov2 = extraer_embeddings(
        rutas_imagenes, dinov2, transformacion_dinov2
    )
    print(f"Extracción en {time.time() - t0:.0f} s")
    guardar_embeddings(embeddings_dinov2, ruta_cache_dinov2)
print(f"Forma de los embeddings: {embeddings_dinov2.shape}")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Sobre el tiempo</strong>

1. `vit_small_patch14_dinov2` tarda bastante más que `resnet18` para las mismas 357 imágenes (la primera vez que corren esta celda, antes de que exista la caché). Miren la tabla que compara los extractores al inicio de la Etapa 2, en particular el tamaño de entrada, los parámetros y la dimensión del embedding: ¿cuáles de esos datos explican la diferencia de tiempo y cuál no?
 </div>


`Escriban su respuesta aquí:`


### 2.3 — Recorte central frente a tiles (0,4 puntos)

Los cuadrantes de pradera son fotos de proporción 2:1 (70 × 30 cm). Un `CenterCrop` cuadrado sobre una imagen así descarta buena parte del ancho — la Etapa 2.1 ya mostró que ese es el eje que más pierde. Midamos cuánto, exactamente, sobre una foto real del dataset.


In [ ]:
imagen_ejemplo = Image.open(rutas_imagenes[0])
ancho_original, alto_original = imagen_ejemplo.size
ancho_recortado = config_dinov2["input_size"][-1]
# El Resize conserva la proporción: la fracción que sobrevive al CenterCrop
# es la misma en la imagen redimensionada que en la original.
ancho_redimensionado = timm.data.create_transform(**config_dinov2).transforms[
    0
](imagen_ejemplo).size[0]
fraccion_conservada = ancho_recortado / ancho_redimensionado
print(f"Imagen original: {ancho_original} x {alto_original} px")
print(f"Tras el Resize: {ancho_redimensionado} px de ancho")
print(f"CenterCrop final: {ancho_recortado} px de ancho")
print(f"Fracción del ancho conservada: {fraccion_conservada:.1%}")

recorte_dinov2 = Compose(transformacion_dinov2.transforms[:2])(imagen_ejemplo)
mostrar_imagenes(
    [imagen_ejemplo, recorte_dinov2],
    [
        "Cuadrante original",
        f"Recorte central de DINOv2 ({recorte_dinov2.width} × {recorte_dinov2.height})",
    ],
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ La mitad del cuadrante</strong>

1. El `CenterCrop` de DINOv2 conserva alrededor de la mitad del ancho del cuadrante. Si el pasto no está distribuido de forma pareja —por ejemplo, hay más trébol en un borde que en el centro—, ¿qué le pasa al embedding del recorte central respecto del cuadrante real?
 </div>


`Escriban su respuesta aquí:`


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Cómo no perder la mitad</strong>

1. El recorte central deja fuera cerca de la mitad del ancho de cada cuadrante. Antes de seguir, propongan al menos una forma de obtener un embedding que represente el cuadrante completo, sin entrenar nada nuevo. ¿Qué costo tendría su propuesta?
 </div>


`Escriban su respuesta aquí:`


Probemos la alternativa: dividir cada imagen en dos tiles cuadrados que juntos cubren el cuadrante completo, extraer un embedding por tile y promediarlos.

Primero, en el notebook: corten `imagen_ejemplo` en dos mitades verticales, una a la izquierda y otra a la derecha, que juntas cubran el ancho completo. Guárdenlas en una lista `tiles_exploracion`. Como la foto es de 1000 × 500 px, cada mitad debe medir 500 × 500 px: un cuadrado.

(Hint: `Image.crop`)


In [ ]:
# Su código aquí
raise NotImplementedError("Construyan tiles_exploracion antes de continuar.")

Para dividir en cualquier número de tiles, y no solo en dos, conviene una función.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `embeddings.py`</strong>

Implementen en <code>src/pasturecam/embeddings.py</code> <code>dividir_en_tiles(imagen, n)</code>: generaliza lo que acaban de explorar. Divide la imagen en <code>n</code> tiles verticales sin solape que cubren el ancho completo (el último absorbe el resto si no divide exacto).

</div>


<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📌 Nota sobre `extraer_embeddings` y rutas de tiles</strong>
<code>extraer_embeddings</code> recibe rutas de archivo, porque así quedó definido su contrato en la Etapa 2.2 — pero un tile vive solo en memoria, nunca se guarda a disco. Por eso la celda de abajo no llama a <code>extraer_embeddings</code> sobre los tiles: aplica la transformación y el extractor directamente a cada uno, y promedia los dos resultados para reemplazar al embedding del recorte central.

</div>


Veamos los tiles de la misma foto de ejemplo con la función. Si `dividir_en_tiles` está bien implementada, son dos cuadrados que juntos cubren el ancho completo del cuadrante, del mismo tamaño que los de la exploración.


In [ ]:
from src.pasturecam.embeddings import dividir_en_tiles

tiles_ejemplo = dividir_en_tiles(imagen_ejemplo, 2)
print(f"Tamaños en la exploración: {[t.size for t in tiles_exploracion]}")
print(f"Tamaños con la función:    {[t.size for t in tiles_ejemplo]}")
mostrar_imagenes(
    [imagen_ejemplo, *tiles_ejemplo],
    ["Cuadrante completo", "Tile izquierdo", "Tile derecho"],
)

In [ ]:
import torch

from src.pasturecam.embeddings import dividir_en_tiles

ruta_cache_tiles = PROCESSED_DIR / "embeddings_dinov2_tiles.parquet"
if ruta_cache_tiles.exists():
    embeddings_tiles = leer_embeddings(ruta_cache_tiles)
else:
    t0 = time.time()
    embeddings_tiles = []
    with torch.no_grad():
        for ruta in rutas_imagenes:
            imagen = Image.open(ruta).convert("RGB")
            tiles = dividir_en_tiles(imagen, 2)
            embeddings_de_tiles = [
                dinov2(transformacion_dinov2(tile).unsqueeze(0)).squeeze(0).numpy()
                for tile in tiles
            ]
            embeddings_tiles.append(np.mean(embeddings_de_tiles, axis=0))
    embeddings_tiles = np.array(embeddings_tiles)
    print(f"Extracción en {time.time() - t0:.0f} s")
    guardar_embeddings(embeddings_tiles, ruta_cache_tiles)
print(f"Forma de los embeddings: {embeddings_tiles.shape}")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
`embeddings_tiles.shape` debe ser `(357, 384)`, la misma dimensión que `embeddings_dinov2` (el promedio no cambia la dimensión, solo combina dos vectores en uno). Con `uv run pytest -m etapa2` deben pasar las cuatro pruebas de la etapa.

</div>


---

Con tres variantes de embeddings guardadas, la Etapa 3 construye el pipeline que va a predecir biomasa a partir de ellas — y con eso, la primera oportunidad real para que algo salga mal sin que se note.


---

## Etapa 3 — Pipelines y leakage (2,0 puntos)


<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📜 "Esto es demasiado bueno"</strong>

El primer modelo del equipo le devuelve a Remigio un R² altísimo, mucho mejor que cualquier cosa que haya visto en años leyendo platos medidores. En vez de alegrarse, desconfía: "si fuera tan fácil, alguien ya lo estaría vendiendo en Osorno". Tiene razón en desconfiar — pero todavía no sabe exactamente por qué.

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Leakage y validación agrupada</strong>

La validación cruzada de la 1.5 usó <code>KFold</code> con <code>shuffle=True</code>, que asigna cada foto a una partición al azar.

Ese "al azar" asume que las observaciones son independientes entre sí. Cuando varias filas comparten algo que el modelo podría aprovechar —vienen del mismo paciente, del mismo edificio, de la misma sesión de medición— esa independencia se rompe. Si una fila de entrenamiento y una de validación son casi gemelas, el modelo no está generalizando: está reconociendo a su gemela. Eso es <strong>leakage</strong>: el entrenamiento contiene información sobre la validación que el modelo no tendría al predecir un caso nuevo, y el puntaje deja de medir lo que dice medir.

<code>GroupKFold</code> corrige este tipo de leakage: recibe una variable de grupo y garantiza que ningún grupo aparezca a la vez en entrenamiento y en validación.

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Regularización: Ridge, Lasso y ElasticNet</strong>

Cuando hay muchas variables y pocas observaciones, una regresión lineal puede ajustar el ruido del entrenamiento con pesos enormes. La <strong>regularización</strong> lo evita sumándole al error una penalización por el tamaño de los pesos: el modelo prefiere pesos chicos, salvo que uno grande mejore mucho el ajuste. Hay dos penalizaciones básicas y una combinación de ambas:

  <ul>
    <li><strong>Ridge (L2)</strong>: penaliza la suma de los pesos al cuadrado, $\alpha \sum_j w_j^2$. Achica todos los pesos, pero casi nunca deja alguno exactamente en 0.</li>
    <li><strong>Lasso (L1)</strong>: penaliza la suma de sus valores absolutos, $\alpha \sum_j |w_j|$. Deja muchos pesos exactamente en 0, así que además elige qué variables usar.</li>
    <li><strong>ElasticNet</strong>: combina las dos. Un parámetro, <code>l1_ratio</code> en scikit-learn, decide cuánto pesa cada una: con 1 es Lasso y con 0, Ridge.</li>
  </ul>

<code>alpha</code> (cuánto se penaliza) y <code>l1_ratio</code> (qué penalización domina) son <strong>hiperparámetros</strong>: valores que no se aprenden de los datos, sino que se fijan antes de ajustar el modelo. Con <code>alpha</code> alto los pesos quedan chicos y el modelo es más simple; con <code>alpha</code> cercano a 0 vuelve a ser una regresión lineal común. En este laboratorio usamos Ridge, y en la 3.4 vamos a mover su <code>alpha</code> a mano. Cómo elegir hiperparámetros de forma sistemática lo veremos en un laboratorio posterior.

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Pipeline y Ridge</strong>

Un <strong>Pipeline</strong> de scikit-learn encadena pasos que se ajustan y se aplican en orden, y se usa como un solo modelo: <code>fit</code> ajusta cada paso con los datos que recibe y <code>predict</code> los aplica en cadena. Dentro de una validación cruzada, el pipeline entero se vuelve a ajustar en cada partición usando solo su entrenamiento, así que ningún paso puede ver datos de la validación.

El pipeline de esta etapa tiene dos pasos:

  <ul>
    <li><code>StandardScaler</code>: lleva cada componente del embedding a media 0 y desviación 1, para que ninguna pese más solo por su escala.</li>
    <li><code>RidgeCV</code>: la regresión Ridge del recuadro anterior, pero sin fijar <code>alpha</code> a mano. Prueba cada valor de <code>alphas_ridge</code> (15 valores entre 0,1 y 10 000, en escala logarítmica) con una validación interna sobre los datos de entrenamiento que recibe, y se queda con el mejor; de ahí la «CV». Como solo ve el entrenamiento de cada partición, esa elección no toca la validación con que medimos el puntaje. Ojo: por omisión, esa validación interna es <em>leave-one-out</em> (deja afuera una foto cada vez) y no conoce las fechas, así que mezcla fotos del mismo día. No contamina el puntaje, pero puede sesgar la elección de <code>alpha</code>; lo retomamos en la 3.4.</li>
  </ul>
</div>


### 3.1 — Leakage por partición: `KFold` vs `GroupKFold` (0,8 puntos)

Los embeddings tienen 384 o 512 dimensiones y solo tenemos 357 fotos. Con más columnas que filas, una regresión lineal sin restricciones puede ajustar el entrenamiento a la perfección y fallar con fotos nuevas. El pipeline lo evita con `RidgeCV`, que penaliza los pesos grandes.

El procedimiento es este:

1. Armar a mano el pipeline `StandardScaler → RidgeCV` y evaluarlo en un solo fold, para ver la validación paso a paso.
2. Llevarlo a dos funciones, `construir_pipeline` y `evaluar`, que repiten ese procedimiento en cada fold.
3. Evaluar con `evaluar` dos veces, cambiando solo cómo se reparten las fotos entre entrenamiento y validación: `KFold(5)`, al azar, y `GroupKFold(5)`, que deja todas las fotos de una misma `Sampling_Date` en la misma partición (la variable de grupo que elegimos en la 1.3).
4. En ambos casos, predecir `Dry_Total_g` y medir con `r2_log`, el R² sobre `log1p` de la 1.5. `evaluar` entrega un `ResultadoCV` con el puntaje de cada partición, su promedio y su desviación.
5. Hacerlo primero con `embeddings_tiles` (DINOv2 con dos tiles, el extractor principal) y después repetirlo con los tres embeddings de la Etapa 2.

El pipeline, los datos y la métrica son los mismos en las dos evaluaciones. Si `KFold` puntúa claramente más que `GroupKFold`, la diferencia solo puede venir de cómo se repartieron las fotos: es la marca del leakage por partición.


In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold, KFold

grupos_fecha = mediciones["Sampling_Date"].to_numpy()
alphas_ridge = np.logspace(-1, 4, 15)

kfold_5 = KFold(n_splits=5, shuffle=True, random_state=0)
groupkfold_5 = GroupKFold(n_splits=5)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Dos formas de repartir las fotos</strong>

1. `kfold_5` reparte las fotos al azar entre las cinco particiones; `groupkfold_5` reparte fechas completas, según `grupos_fecha`. Si dos fotos se tomaron el mismo día, ¿pueden quedar una en entrenamiento y la otra en validación con `kfold_5`? ¿Y con `groupkfold_5`?
2. Antes de evaluar nada: ¿con cuál de las dos esperan un puntaje más alto, y por qué?
 </div>


`Escriban sus respuestas aquí:`


Comprobemos cuántas fechas quedan repartidas a ambos lados en el primer fold de cada validación.


In [ ]:
indices_train, indices_val = next(kfold_5.split(embeddings_tiles))
compartidas_kfold = set(grupos_fecha[indices_train]) & set(grupos_fecha[indices_val])
indices_train, indices_val = next(groupkfold_5.split(embeddings_tiles, y_total, grupos_fecha))
compartidas_grupal = set(grupos_fecha[indices_train]) & set(grupos_fecha[indices_val])
print(f"Primer fold de KFold:      {len(compartidas_kfold)} fechas con fotos en entrenamiento y en validación")
print(f"Primer fold de GroupKFold: {len(compartidas_grupal)} fechas con fotos en entrenamiento y en validación")

Primero el pipeline. Armen `pipeline_explorado` con `Pipeline` de scikit-learn: un `StandardScaler` y un `RidgeCV(alphas=alphas_ridge)`, en ese orden y sin ajustarlo.

(Hint: `Pipeline`)


In [ ]:
# Su código aquí
raise NotImplementedError("Construyan pipeline_explorado antes de continuar.")

Ahora evalúenlo en un solo fold. Tomen la primera partición de `kfold_5.split(embeddings_tiles)`: ajusten el pipeline **solo** con las fotos de entrenamiento, predigan las de validación y calculen `r2_log` sobre ellas. Guarden el resultado en `puntaje_fold`.

(Hint: `next`, `fit`, `predict`)


In [ ]:
# Su código aquí
raise NotImplementedError("Calculen puntaje_fold antes de continuar.")

Ese procedimiento se repite en cada fold y con distintas validaciones, así que conviene tenerlo en funciones.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `evaluation.py`</strong>

Implementen en <code>src/pasturecam/evaluation.py</code>:

  <ul>
    <li><code>construir_pipeline(regresor)</code>: el mismo pipeline que armaron a mano, sin ajustarlo, con dos pasos llamados <code>"escalador"</code> (un <code>StandardScaler</code>) y <code>"regresor"</code> (el <code>regresor</code> recibido). La prueba revisa esos nombres.</li>
    <li><code>evaluar(pipeline, X, y, grupos, cv, metrica=r2_score)</code>: lo que hicieron en un fold, pero en todos. Por cada fold de <code>cv.split(X, y, grupos)</code>, clona el pipeline, lo ajusta <strong>solo</strong> con el fold de entrenamiento y mide <code>metrica</code> sobre el de validación. Devuelve un <code>ResultadoCV</code> con los puntajes por fold, su promedio, su desviación estándar (<code>np.std</code>) y <code>descripcion_validacion</code> con el formato <code>"&lt;Clase&gt;(&lt;n&gt;)"</code>: el nombre de la clase de <code>cv</code> y su número de particiones, por ejemplo <code>"GroupKFold(5)"</code>. La prueba y la tabla de la Etapa 5 dependen de ese formato.</li>
  </ul>
</div>


In [ ]:
import warnings

from src.pasturecam.evaluation import construir_pipeline, evaluar

# `evaluar` le pasa `grupos` a cualquier validación. `KFold` los ignora (esa es
# justamente su diferencia con `GroupKFold`) y scikit-learn lo avisa con una
# UserWarning; la silenciamos porque aquí ese comportamiento es el esperado.
warnings.filterwarnings("ignore", message="The groups parameter is ignored")

pipeline_tiles = construir_pipeline(RidgeCV(alphas=alphas_ridge))
resultado_kfold = evaluar(
    pipeline_tiles, embeddings_tiles, y_total, grupos_fecha, kfold_5, metrica=r2_log
)
print(f"Primer fold, a mano:   R2_log = {puntaje_fold:.3f}")
print(f"Primer fold, evaluar:  R2_log = {resultado_kfold.puntajes[0]:.3f}")
print(f"KFold (promedio):      R2_log = {resultado_kfold.promedio:.3f} +/- {resultado_kfold.desviacion:.3f}")

Ese número ya parece bastante mejor que los baselines de la Etapa 1. Antes de festinar, repitamos exactamente el mismo pipeline con la variable de grupo que identificamos en la 1.3.


In [ ]:
resultado_grupal = evaluar(
    pipeline_tiles, embeddings_tiles, y_total, grupos_fecha, groupkfold_5, metrica=r2_log
)
print(f"GroupKFold: R2_log = {resultado_grupal.promedio:.3f} +/- {resultado_grupal.desviacion:.3f}")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
El primer fold a mano y el de `evaluar` deben dar el mismo puntaje, y `resultado_kfold.promedio` debe ser mayor que `resultado_grupal.promedio` (cercanos a 0,44 y 0,31). Con `uv run pytest -m etapa3` deben pasar las cuatro pruebas de la etapa.

</div>


Una sola comparación podría ser casualidad. Repitamos el mismo contraste con los tres embeddings de la Etapa 2, para ver si el patrón se mantiene.


Cada evaluación la registramos en MLflow con `registrar_run`, con un nombre estable por variante y validación. Las de `GroupKFold` guardan además el pipeline ajustado con todas las fotos, para poder recargarlo en la Etapa 5.


In [ ]:
variantes = {
    "resnet18, centro": embeddings_resnet,
    "DINOv2, centro": embeddings_dinov2,
    "DINOv2, 2 tiles": embeddings_tiles,
}

for nombre, X in variantes.items():
    pipeline = construir_pipeline(RidgeCV(alphas=alphas_ridge))
    r_kfold = evaluar(pipeline, X, y_total, grupos_fecha, kfold_5, metrica=r2_log)
    r_grupal = evaluar(pipeline, X, y_total, grupos_fecha, groupkfold_5, metrica=r2_log)
    params = {"modelo": "RidgeCV", "entrada": nombre}
    clave = nombre.replace(", ", "_").replace(" ", "-")
    registrar_run(f"ridge_{clave}_kfold", r_kfold, params, etiquetas={"tipo": "modelo"})
    registrar_run(
        f"ridge_{clave}_groupkfold",
        r_grupal,
        params,
        modelo=construir_pipeline(RidgeCV(alphas=alphas_ridge)).fit(X, y_total),
        etiquetas={"tipo": "modelo"},
    )
    print(
        f"{nombre:18s}  KFold={r_kfold.promedio:+.3f}  GroupKFold={r_grupal.promedio:+.3f}"
        f"  brecha={r_kfold.promedio - r_grupal.promedio:+.3f}"
    )

En los dos embeddings de DINOv2, `KFold` puntúa claramente más que `GroupKFold`: 0,32 frente a 0,10 con recorte central y 0,44 frente a 0,31 con tiles. El pipeline, los datos y la métrica son los mismos; lo único que cambió fue cómo se repartieron las fotos. **En DINOv2 hay leakage por partición.** En `resnet18` el patrón no aparece —ahí `KFold` puntúa incluso más bajo—, y la pregunta de abajo trata de esa diferencia.


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Una brecha que no es igual en los tres</strong>

1. En DINOv2 (centro y tiles), `KFold` puntúa más alto que `GroupKFold`, como esperaba Remigio que desconfiara. En `resnet18`, pasa al revés. ¿Qué tienen en común las fotos tomadas el mismo día que `KFold` no distingue y `GroupKFold` sí, y por qué esa ventaja le serviría más a un embedding que describe mejor la textura del pasto que a uno que no?
2. Si Remigio usara el modelo de DINOv2 con tiles para decidir la carga animal de un potrero nuevo, en una fecha que el modelo nunca vio, ¿cuál de los dos números —`KFold` o `GroupKFold`— se parece más a lo que le va a pasar en la práctica?
 </div>


`Escriban sus respuestas aquí:`


Para comparar la foto con el plato en la Etapa 5 con la misma vara, registremos también los dos baselines de la 1.5, ahora con la validación agrupada por fecha.


In [ ]:
from sklearn.dummy import DummyRegressor

resultado_media_grupal = evaluar(
    DummyRegressor(strategy="mean"), altura, y_total, grupos_fecha, groupkfold_5, metrica=r2_log
)
registrar_run(
    "baseline-media_groupkfold",
    resultado_media_grupal,
    {"modelo": "DummyRegressor", "entrada": "ninguna"},
    etiquetas={"tipo": "baseline"},
)
resultado_altura_grupal = evaluar(
    LinearRegression(), altura, y_total, grupos_fecha, groupkfold_5, metrica=r2_log
)
registrar_run(
    "baseline-altura_groupkfold",
    resultado_altura_grupal,
    {"modelo": "LinearRegression", "entrada": "Height_Ave_cm"},
    modelo=LinearRegression().fit(altura, y_total),
    etiquetas={"tipo": "baseline"},
)
print(f"Media (Dummy), GroupKFold:  R2_log = {resultado_media_grupal.promedio:+.3f}")
print(f"Altura (plato), GroupKFold: R2_log = {resultado_altura_grupal.promedio:+.3f}")

### 3.2 — Leakage por componentes del objetivo (0,4 puntos)

`Dry_Total_g` es, por construcción, `Dry_Green_g + Dry_Dead_g + Dry_Clover_g`. ¿Qué pasa si una de esas componentes se usa como feature para predecir el total?


In [ ]:
X_con_fuga = np.hstack([embeddings_tiles, mediciones[["Dry_Green_g"]].to_numpy()])

resultado_sin_fuga = evaluar(
    construir_pipeline(RidgeCV(alphas=alphas_ridge)),
    embeddings_tiles,
    y_total,
    grupos_fecha,
    groupkfold_5,
    metrica=r2_log,
)
resultado_con_fuga = evaluar(
    construir_pipeline(RidgeCV(alphas=alphas_ridge)),
    X_con_fuga,
    y_total,
    grupos_fecha,
    groupkfold_5,
    metrica=r2_log,
)
print(f"Sin Dry_Green_g: R2_log = {resultado_sin_fuga.promedio:.3f}")
print(f"Con Dry_Green_g: R2_log = {resultado_con_fuga.promedio:.3f}")

# Lo registramos marcado como leakage: en producción no existe Dry_Green_g.
registrar_run(
    "ridge_DINOv2_2-tiles_con-dry-green_groupkfold",
    resultado_con_fuga,
    {"modelo": "RidgeCV", "entrada": "DINOv2, 2 tiles + Dry_Green_g"},
    etiquetas={"tipo": "leakage"},
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Sube, pero no a casi perfecto</strong>

1. El puntaje sube con claridad al agregar `Dry_Green_g` (de 0,31 a 0,47), pero no se acerca a 1,0. `Dry_Total_g` es la suma de tres componentes: ¿por qué conocer solo una de las tres no alcanza para predecir el total casi perfectamente?
2. En el potrero, cuando el modelo tenga que predecir la biomasa de una foto nueva, ¿va a estar disponible `Dry_Green_g`? ¿Por qué este modelo, aunque puntúe mejor, no sirve para lo que Remigio necesita?
 </div>


`Escriban sus respuestas aquí:`


### 3.3 — ¿La altura es leakage o una feature legítima? (0,3 puntos)

La Etapa 1 usó `Height_Ave_cm` como baseline. Agregarla junto a los embeddings de la foto podría mejorar el puntaje, pero antes de usarla hay que preguntarse si tiene sentido exigirla en la decisión real de Remigio.


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Altura: ¿leakage o feature?</strong>

1. A diferencia de `Dry_Green_g`, `Height_Ave_cm` sí se puede medir en el momento, en el potrero. ¿Es entonces leakage, como `Dry_Green_g`? Aunque no lo sea, ¿de qué depende que le sirva a Remigio como variable de entrada?
 </div>


`Escriban su respuesta aquí:`


### 3.4 — Un adelanto: el `alpha` de Ridge (0,5 puntos)

En la 3.1, `RidgeCV` eligió `alpha` por su cuenta. `alpha` es un hiperparámetro, como vimos en el recuadro de regularización. Cómo elegirlos bien es tema de un laboratorio más adelante; acá solo vamos a mover `alpha` a mano y mirar qué pasa con el mismo pipeline de la 3.1, usando `Ridge` en vez de `RidgeCV` y la misma validación `GroupKFold` por fecha.


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Antes de mover `alpha`</strong>

1. ¿Qué esperan que pase con el puntaje si `alpha` es muy chico (0,01) y si es enorme (100 000)? Piensen en qué les pasa a los pesos del modelo en cada extremo.
 </div>


`Escriban su respuesta aquí:`


Prueben distintos valores: editen `alphas_a_probar` y vuelvan a ejecutar la celda. Cada valor queda registrado en MLflow como exploración.


In [ ]:
from sklearn.linear_model import Ridge

alphas_a_probar = [0.01, 1, 100, 1_000, 10_000, 100_000]

for alpha in alphas_a_probar:
    pipeline_alpha = construir_pipeline(Ridge(alpha=alpha))
    resultado_alpha = evaluar(
        pipeline_alpha, embeddings_tiles, y_total, grupos_fecha, groupkfold_5, metrica=r2_log
    )
    registrar_run(
        f"ridge_alpha-{alpha}_groupkfold",
        resultado_alpha,
        {"modelo": "Ridge", "alpha": alpha, "entrada": "DINOv2, 2 tiles"},
        etiquetas={"tipo": "exploracion"},
    )
    print(f"alpha = {alpha:>9}   R2_log = {resultado_alpha.promedio:+.3f}")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Elegir mirando el puntaje</strong>

1. ¿Con qué `alpha` obtuvieron el mejor puntaje? Si eligieran ese `alpha` mirando esta tabla y después reportaran ese mismo puntaje como lo que Remigio puede esperar en un potrero nuevo, ¿qué idea de esta etapa estarían pasando por alto?
 </div>


`Escriban su respuesta aquí:`


---

El pipeline ya distingue un número honesto de uno optimista. Falta decidir qué número reportarle al directorio — el tema de la Etapa 4.


---

## Etapa 4 — Métricas (1,0 puntos)


<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📜 "¿Cuántos kilos por hectárea se equivoca?"</strong>

El R² agrupado convenció a Remigio de seguir adelante, pero cuando lo llevó al directorio de la cooperativa, nadie supo qué hacer con un número entre 0 y 1. La pregunta que le hicieron fue otra: "¿cuántos kilos de pasto por hectárea se equivoca el modelo? Así sabemos si afecta la decisión de carga animal o no."

</div>


Para presentar las métricas, supongamos que un modelo predice el tiempo de viaje de un micro en Santiago, en minutos, y lo comparamos contra el tiempo real:

| Viaje | Real | Predicho | Error |
| ----- | ---- | -------- | ----- |
| A     | 20   | 22       | 2     |
| B     | 15   | 12       | −3    |
| C     | 40   | 55       | 15    |


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 MAE</strong>

El <strong>MAE</strong> (<em>mean absolute error</em>) promedia el valor absoluto de los errores:

$$\text{MAE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i|$$

En el ejemplo, (2 + 3 + 15) / 3 = 6,7 minutos. Es fácil de explicar —«en promedio, el modelo se equivoca por 6,7 minutos»— y le da a cada viaje el mismo peso.

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 RMSE</strong>

El <strong>RMSE</strong> (<em>root mean squared error</em>) eleva cada error al cuadrado antes de promediar y después vuelve a sacar la raíz:

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_i (y_i - \hat{y}_i)^2}$$

En el ejemplo, $\sqrt{(2^2 + 3^2 + 15^2)/3} \approx 8{,}9$ minutos. Al elevar al cuadrado, el viaje C, con un error de 15 minutos, pesa muchísimo más que A o B. El RMSE siempre es mayor o igual que el MAE, y la brecha entre ambos avisa si hay errores grandes aislados.

</div>


<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 R² como métrica</strong>

El <strong>R²</strong>, que definimos en la 1.4, compara el error del modelo con el de predecir siempre el promedio: vale 1,0 si las predicciones son perfectas, 0,0 si el modelo no hace nada mejor que predecir el promedio, y puede ser negativo si hace algo peor que eso. A diferencia del MAE y el RMSE, no tiene unidades, así que no dice cuántos minutos (o gramos) se equivoca el modelo.

</div>


### 4.1 — MAE, RMSE y R² en gramos y en kg/ha (0,4 puntos)

Evaluemos el mejor pipeline de la Etapa 3 (DINOv2 con tiles, `GroupKFold` por fecha) con las tres métricas, sobre las predicciones en gramos por cuadrante.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

predicciones_oof = np.zeros_like(y_total)
for indices_train, indices_test in groupkfold_5.split(
    embeddings_tiles, y_total, grupos_fecha
):
    modelo_fold = construir_pipeline(RidgeCV(alphas=alphas_ridge))
    modelo_fold.fit(embeddings_tiles[indices_train], y_total[indices_train])
    predicciones_oof[indices_test] = modelo_fold.predict(
        embeddings_tiles[indices_test]
    )

mae_g = mean_absolute_error(y_total, predicciones_oof)
rmse_g = mean_squared_error(y_total, predicciones_oof) ** 0.5
r2_plano = r2_score(y_total, predicciones_oof)
print(f"MAE:  {mae_g:.1f} g")
print(f"RMSE: {rmse_g:.1f} g")
print(f"R2:   {r2_plano:.3f}")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Un error muy grande</strong>

1. El MAE es 12,9 g y el RMSE, 17,7 g. Supongan que en un solo cuadrante el modelo se equivoca por 200 g, por ejemplo porque la medición se digitó mal. ¿Cuál de las dos métricas cambiaría más, y por qué? Pueden comprobarlo modificando una predicción de `predicciones_oof` y recalculando.
 </div>


`Escriban su respuesta aquí:`


Esos errores están en gramos por cuadrante, pero el directorio piensa en kilos por hectárea. Conviertan en el notebook el MAE y el RMSE a kg/ha y guárdenlos en `mae_kg_ha` y `rmse_kg_ha`. Un cuadrante de 70 × 30 cm cubre 0,21 m² y una hectárea tiene 10 000 m².

(Hint: regla de tres, y 1 kg = 1000 g)


In [ ]:
# Su código aquí
raise NotImplementedError("Construyan mae_kg_ha y rmse_kg_ha antes de continuar.")

Vamos a convertir unidades varias veces, así que conviene una función.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `metrics.py`</strong>

Implementen en <code>src/pasturecam/metrics.py</code> <code>gramos_a_kg_ha(gramos)</code>: la misma conversión que acaban de explorar. Convierte gramos por cuadrante de 70 × 30 cm a kilogramos por hectárea.

</div>


In [ ]:
from src.pasturecam.metrics import gramos_a_kg_ha

print(f"MAE en kg/ha, con la función:  {gramos_a_kg_ha(mae_g):.0f}")
print(f"RMSE en kg/ha, con la función: {gramos_a_kg_ha(rmse_g):.0f}")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
Ejecutar <code>uv run pytest -m etapa4 -k gramos_a_kg_ha</code>. Los valores de la exploración y los de la función deben coincidir, y <code>gramos_a_kg_ha(1)</code> debe dar aproximadamente 47,6: un gramo en un cuadrante de 0,21 m² equivale a esa cantidad repartida en una hectárea de 10 000 m².

</div>


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Cambiar de unidad</strong>

1. El MAE de 12,9 g por cuadrante equivale a unos 615 kg/ha. ¿La conversión cambia el tamaño del error en relación con la biomasa típica de un cuadrante (unos 45 g)? Justifiquen con un cálculo.
 </div>


`Escriban su respuesta aquí:`


### 4.2 — Evaluar los cinco objetivos a la vez (0,4 puntos)

Hasta ahora hemos trabajado y evaluado sobre la biomasa total, `Dry_Total_g`. Sin embargo, podríamos querer evaluar los cinco objetivos a la vez, cada uno sobre `log1p` y ponderado según su importancia relativa:

- `Dry_Green_g`, `Dry_Dead_g` y `Dry_Clover_g`: 0,1 cada uno;
- `GDM_g`: 0,2;
- `Dry_Total_g`: 0,5.

Entrenemos un regresor por objetivo y calculemos esa métrica ponderada.


In [ ]:
from sklearn.multioutput import MultiOutputRegressor

from src.pasturecam.contratos import PESOS_OBJETIVOS

OBJETIVOS_MODELO = list(PESOS_OBJETIVOS.keys())
Y_multi = mediciones.select(OBJETIVOS_MODELO).to_numpy()

predicciones_multi_oof = np.zeros_like(Y_multi)
for indices_train, indices_test in groupkfold_5.split(
    embeddings_tiles, Y_multi, grupos_fecha
):
    modelo_multi = construir_pipeline(MultiOutputRegressor(RidgeCV(alphas=alphas_ridge)))
    modelo_multi.fit(embeddings_tiles[indices_train], Y_multi[indices_train])
    predicciones_multi_oof[indices_test] = modelo_multi.predict(
        embeddings_tiles[indices_test]
    )

y_real_dict = {
    nombre: Y_multi[:, i] for i, nombre in enumerate(OBJETIVOS_MODELO)
}
y_pred_dict = {
    nombre: predicciones_multi_oof[:, i]
    for i, nombre in enumerate(OBJETIVOS_MODELO)
}

`y_real_dict` y `y_pred_dict` guardan, para cada objetivo, los valores reales y los predichos. Calculen en el notebook la métrica ponderada: para cada objetivo, el R² entre `log1p` de lo real y `log1p` de lo predicho (recortando a 0 las predicciones negativas antes del logaritmo, porque una biomasa negativa no tiene sentido), multiplicado por su peso en `PESOS_OBJETIVOS`. Súmenlos en `puntaje_ponderado`.

(Hint: `np.log1p`, `np.clip`, `r2_score`)


In [ ]:
# Su código aquí
raise NotImplementedError("Construyan puntaje_ponderado antes de continuar.")

Esta métrica la vamos a calcular con distintos modelos, así que la llevamos a una función.

<!-- MODULARIZACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🛠️ Modularización — `metrics.py`</strong>

Implementen en <code>src/pasturecam/metrics.py</code> <code>r2_log_ponderado(y_real, y_pred, pesos=PESOS_OBJETIVOS)</code>: la misma métrica que acaban de explorar. <code>y_real</code> e <code>y_pred</code> son diccionarios <code>{nombre_objetivo: valores}</code>. Calcula el R² sobre <code>log1p</code> de cada objetivo (recortando a 0 antes del logaritmo) y combina los cinco con <code>pesos</code>.

</div>


In [ ]:
from src.pasturecam.metrics import r2_log_ponderado

print(f"R2_log ponderado, con la función: {r2_log_ponderado(y_real_dict, y_pred_dict):.3f}")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
Ejecutar <code>uv run pytest -m etapa4</code>: deben pasar las cinco pruebas de la etapa. Los dos valores impresos arriba deben ser iguales.

</div>


### 4.3 — Qué métrica reportar (0,2 puntos)

Repasemos lo que tenemos: MAE y RMSE en gramos (fáciles de calcular, difíciles de interpretar sin convertir), su versión en kg/ha (la unidad que el directorio pidió), el R² simple, y el R² log1p ponderado de la 4.2 (pensado para comparar entre objetivos de escalas distintas).


<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ La métrica para Remigio</strong>

1. De las métricas calculadas en esta etapa, ¿cuál le propondrían reportar al directorio para decidir si financian la campaña de muestreo, y cuál usarían ustedes internamente para comparar modelos entre sí en las próximas etapas? Justifiquen que no tienen por qué ser la misma.
 </div>


`Escriban su respuesta aquí:`


---

Con una métrica para el directorio y otra para comparar modelos, falta ordenar los experimentos y llegar a una recomendación: el tema de la Etapa 5.


---

## Etapa 5 — Consultar y decidir (1,0 puntos)


<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📜 La última pregunta de Remigio</strong>

Entre embeddings, modelos y validaciones, el equipo ya corrió varios experimentos, y todos quedaron registrados en MLflow. Remigio no quiere una lista de números en la pantalla de un notebook que nadie va a volver a abrir: quiere una recomendación, y quiere poder confiar en que, si alguien la revisa en seis meses, los números todavía se puedan reproducir.

</div>


### 5.1 — La tabla de runs (0,2 puntos)

Todo lo que evaluamos quedó en MLflow. Consultemos la tabla completa, ordenada de mayor a menor puntaje.


In [ ]:
tabla = tabla_runs()
tabla[columnas_runs].sort_values("metrics.promedio", ascending=False)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ El mejor puntaje de la tabla</strong>

1. ¿Deberían recomendarle a Remigio el run con mayor puntaje de toda la tabla? Revisen su `tipo` y su `validacion`, y los de los runs que le siguen: ¿qué tienen en común los que quedan más arriba?
 </div>


`Escriban su respuesta aquí:`


Para decidir, nos quedamos solo con los modelos y los baselines evaluados con `GroupKFold`.


In [ ]:
tabla_decision = tabla[
    tabla["tags.tipo"].isin(["modelo", "baseline"])
    & tabla["tags.validacion"].str.startswith("GroupKFold")
].sort_values("metrics.promedio", ascending=False)
tabla_decision[columnas_runs]

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
<code>tabla</code> debe tener una fila por nombre de run, aunque hayan ejecutado el notebook más de una vez. <code>tabla_decision</code> debe tener cinco filas: los tres embeddings y los dos baselines, todos con <code>GroupKFold</code>.

</div>


### 5.2 — Recargar un run (0,2 puntos)

Los pipelines de la 3.1 evaluados con `GroupKFold` quedaron guardados como artefacto. Recarguemos el de DINOv2 con tiles y comprobemos que predice lo mismo que ese pipeline ajustado de nuevo.


In [ ]:
import mlflow.sklearn

fila_tiles = tabla_decision[
    tabla_decision["tags.mlflow.runName"] == "ridge_DINOv2_2-tiles_groupkfold"
].iloc[0]
modelo_recargado = mlflow.sklearn.load_model(f"runs:/{fila_tiles['run_id']}/modelo")
modelo_de_nuevo = construir_pipeline(RidgeCV(alphas=alphas_ridge)).fit(embeddings_tiles, y_total)
iguales = np.allclose(
    modelo_recargado.predict(embeddings_tiles), modelo_de_nuevo.predict(embeddings_tiles)
)
print(f"Predicciones iguales a las del pipeline ajustado de nuevo: {iguales}")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ Qué vale la pena registrar</strong>

1. `registrar_run` guarda el pipeline completo como artefacto, no solo sus métricas. ¿Qué pueden hacer con el artefacto que no podrían hacer solo con la fila de la fila de `tabla_runs()`?
 </div>


`Escriban su respuesta aquí:`


### 5.3 — La recomendación para Remigio (0,6 puntos)

Cerremos el encargo con la comparación que de verdad le importa, leída de la tabla de decisión: la mejor foto contra el plato, los dos evaluados con la misma validación agrupada por fecha.


In [ ]:
fila_plato = tabla_decision[
    tabla_decision["tags.mlflow.runName"] == "baseline-altura_groupkfold"
].iloc[0]
mejor_foto = tabla_decision[tabla_decision["tags.tipo"] == "modelo"].iloc[0]
print(f"Plato medidor (altura), GroupKFold:  R2_log = {fila_plato['metrics.promedio']:+.3f}")
print(f"Mejor modelo con foto ({mejor_foto['tags.mlflow.runName']}): R2_log = {mejor_foto['metrics.promedio']:+.3f}")

El directorio pidió el error en kilos por hectárea. El de la foto ya lo calculamos en la 4.1; calculemos el del plato con la misma validación agrupada.


In [ ]:
predicciones_plato_oof = np.zeros_like(y_total)
for indices_train, indices_test in groupkfold_5.split(
    altura, y_total, grupos_fecha
):
    modelo_plato = LinearRegression()
    modelo_plato.fit(altura[indices_train], y_total[indices_train])
    predicciones_plato_oof[indices_test] = modelo_plato.predict(
        altura[indices_test]
    )

mae_plato_g = mean_absolute_error(y_total, predicciones_plato_oof)
print(f"MAE del plato medidor:    {gramos_a_kg_ha(mae_plato_g):.0f} kg/ha")
print(f"MAE de DINOv2 con tiles:  {gramos_a_kg_ha(mae_g):.0f} kg/ha")

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">❓ La recomendación final</strong>

1. Con los números de arriba, y con el margen entre ellos, ¿qué le recomiendan a Remigio: reemplazar el plato por una foto, mantener el plato, o financiar una campaña propia de fotos y mediciones en Osorno antes de decidir? Justifiquen citando los números de este notebook, no una intuición.
 </div>


`Escriban su respuesta aquí:`


---

¡Listo! Remigio tiene un número honesto, una métrica que su directorio entiende, varios modelos comparados con la misma vara y una recomendación con los runs que la respaldan. Las dudas se resuelven en el foro del curso en U-Cursos, y la entrega, con la Pull Request abierta, vence el domingo siguiente a esta sesión a las 23:59.
